# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SUKRIT004/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will use Random Forest classification. The target is whether a page is observed as declining, represented by is_declining_label. Random Forest fits this task because it can capture nonlinear relationships between search performance, traffic, engagement, and content signals without requiring a linear relationship. It also provides feature importance that can help explain what signals the model relies on. I will compare it with the Week-4 transparent baseline using the same evaluation data and ranking metric. Model complexity is only useful if it improves the decision-support ranking.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a client-level holdout split, with approximately 80% of clients used for training and 20% held out for evaluation. This prevents pages from the same client appearing in both training and evaluation and gives a more realistic estimate of performance on unseen clients. The baseline and Random Forest are evaluated on the same held-out rows using the same Precision@50 metric.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The Random Forest achieved a Precision@50 of 0.82, while the Week-4 baseline achieved 0.84 on the same held-out clients. The baseline therefore performed slightly better, with a difference of 0.02 Precision@50. The model's ROC-AUC was 0.6142, showing some ability to distinguish declining from non-declining pages, but this did not translate into a better top-50 ranking. I therefore do not prefer the more complex model solely because it uses ML.

In [11]:
# ML-08 — Train Random Forest and compare with baseline

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# ---------------------------------------------------------
# 1. Define target, groups and honest features
# ---------------------------------------------------------

target = "is_declining_label"
group = "client_id"

numeric_features = [
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "search_volume",
    "competition",
    "cpc"
]

categorical_features = [
    "content_type",
    "main_intent",
    "competition_level",
    "age_tier",
    "freshness_tier",
    "position_tier",
    "impression_tier"
]

numeric_features = [
    c for c in numeric_features
    if c in data.columns
]

categorical_features = [
    c for c in categorical_features
    if c in data.columns
]

features = numeric_features + categorical_features

X = data[features].copy()
y = data[target].astype(int)
groups = data[group]

print("Features used:", len(features))
print(features)

# ---------------------------------------------------------
# 2. Client-level train/test split
# ---------------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("\nTrain rows:", len(X_train))
print("Test rows:", len(X_test))

print(
    "Train clients:",
    data.iloc[train_idx][group].nunique()
)

print(
    "Test clients:",
    data.iloc[test_idx][group].nunique()
)

# ---------------------------------------------------------
# 3. Preprocessing
# ---------------------------------------------------------

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# ---------------------------------------------------------
# 4. Random Forest
# ---------------------------------------------------------

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

print("\nTraining Random Forest...")
pipeline.fit(X_train, y_train)

# ---------------------------------------------------------
# 5. Predictions
# ---------------------------------------------------------

test_probability = pipeline.predict_proba(X_test)[:, 1]

print(
    "\nROC-AUC:",
    round(roc_auc_score(y_test, test_probability), 4)
)

# ---------------------------------------------------------
# 6. Precision@50
# ---------------------------------------------------------

evaluation = data.iloc[test_idx].copy()

evaluation["model_score"] = test_probability

evaluation = evaluation.sort_values(
    "model_score",
    ascending=False
)

top50 = evaluation.head(50)

model_precision_at_50 = top50[target].mean()

print(
    "Model Precision@50:",
    round(model_precision_at_50, 4)
)

# ---------------------------------------------------------
# 7. Baseline on EXACT SAME test rows
# ---------------------------------------------------------

# Recreate the Week-4 baseline using only the held-out rows.
# Expected CTR is calculated from TRAINING data only.

train_reference = data.iloc[train_idx].copy()

expected_ctr = (
    train_reference
    .groupby("position_tier")["ctr"]
    .mean()
)

evaluation["expected_ctr"] = (
    evaluation["position_tier"]
    .map(expected_ctr)
)

evaluation["ctr_gap"] = (
    evaluation["expected_ctr"] -
    evaluation["ctr"]
).clip(lower=0)

evaluation["baseline_score"] = (
    evaluation["ctr_gap"] *
    np.log1p(evaluation["impressions_90d"])
)

baseline_evaluation = evaluation.sort_values(
    "baseline_score",
    ascending=False
)

baseline_top50 = baseline_evaluation.head(50)

baseline_precision_at_50 = (
    baseline_top50[target].mean()
)

print(
    "Baseline Precision@50:",
    round(baseline_precision_at_50, 4)
)

# ---------------------------------------------------------
# 8. Model vs baseline table
# ---------------------------------------------------------

comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Random Forest"
    ],
    "precision_at_50": [
        baseline_precision_at_50,
        model_precision_at_50
    ]
})

print("\n=== MODEL VS BASELINE ===")
display(comparison)

print(
    "\nLift:",
    round(
        model_precision_at_50 -
        baseline_precision_at_50,
        4
    )
)

Features used: 25
['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'search_volume', 'competition', 'cpc', 'content_type', 'main_intent', 'competition_level', 'age_tier', 'freshness_tier', 'position_tier', 'impression_tier']

Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7

Training Random Forest...

ROC-AUC: 0.6142
Model Precision@50: 0.82
Baseline Precision@50: 0.84

=== MODEL VS BASELINE ===


,method,precision_at_50
0,Week-4 baseline,0.84
1,Random Forest,0.82



Lift: -0.02


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The Random Forest achieved a Precision@50 of 0.82 compared with 0.84 for the transparent Week-4 baseline, so the model did not outperform the simpler rule on this held-out client set. The ROC-AUC was 0.6142, indicating that the model has some ability to distinguish declining from non-declining pages, but its ranking of the highest-priority 50 pages was slightly worse than the baseline. This suggests that model complexity alone did not improve the decision-support task. Some errors may occur because the available features do not fully capture query intent, content quality, or other factors that influence observed search performance. The result is directional and specific to this validation setup.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 — Feature importance

feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
importances = pipeline.named_steps["model"].feature_importances_

importance_df = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    })
    .sort_values("importance", ascending=False)
    .head(15)
)

print("Top model features:")
display(importance_df)

# Top-50 model errors

error_check = evaluation.copy()

error_check["predicted_positive"] = (
    error_check["model_score"] >= evaluation["model_score"].nlargest(50).min()
).astype(int)

false_positive = error_check[
    (error_check["predicted_positive"] == 1) &
    (error_check[target] == 0)
].head(10)

false_negative = error_check[
    (error_check["predicted_positive"] == 0) &
    (error_check[target] == 1)
].head(10)

print("False positives among top-50:")
display(
    false_positive[
        ["content_type", "impressions_90d", "ctr",
         "avg_position", "content_age_days", target]
    ]
)

print("\nExample false negatives outside top-50:")
display(
    false_negative[
        ["content_type", "impressions_90d", "ctr",
         "avg_position", "content_age_days", target]
    ]
)

Top model features:


,feature,importance
0,num__impressions_90d,0.166762
8,num__content_age_days,0.124603
11,num__avg_position,0.118704
13,num__scroll_rate,0.046992
10,num__ctr,0.043032
1,num__clicks_90d,0.034800
42,cat__position_tier_top_3,0.033761
2,num__pageviews_90d,0.032369
32,cat__age_tier_365+,0.032340
3,num__sessions_90d,0.029731


False positives among top-50:


,content_type,impressions_90d,ctr,avg_position,content_age_days,is_declining_label
11376,keyword article,8979,0.17,19.6,95,0
27993,keyword article,1266,0.00,4.6,106,0
17602,keyword article,4650,0.09,28.5,97,0
20389,keyword article,1152,0.35,2.7,117,0
22812,keyword article,701,0.00,12.8,95,0
4545,keyword article,2498,0.12,6.5,138,0
3488,keyword article,13502,0.02,19.7,106,0
21828,keyword article,3813,0.08,11.1,106,0
4827,keyword article,384,0.26,22.6,141,0



Example false negatives outside top-50:


,content_type,impressions_90d,ctr,avg_position,content_age_days,is_declining_label
3100,keyword article,442,0.00,10.0,117,1
3081,keyword article,805,0.00,24.0,95,1
16761,keyword article,3322,0.18,14.2,97,1
18962,keyword article,4122,0.12,21.6,95,1
24126,keyword article,384,0.00,18.0,95,1
22,keyword article,2311,0.13,3.9,228,1
18042,keyword article,462,0.22,14.8,117,1
4140,keyword article,1801,0.11,30.4,228,1
29866,keyword article,2243,0.04,24.2,106,1
7954,keyword article,621,0.16,3.9,117,1


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.